# Value at Risk

Weekly total returns from `spx_returns_weekly.xlsx` (sheet `spx returns`), sample **2015-07-10 to 2026-07-03**
(574 weekly observations). Four securities are evaluated: **AAPL, META, NVDA, TSLA**.

Conventions used throughout:

- $q = 0.05$ is the significance level.
- Volatilities are annualized with $\sqrt{52}$.
- $M = 26$ weeks is the rolling window used in Section 2.
- VaR and CVaR are reported **as returns**, so a loss is a negative number.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

# Resolve the workbook from the notebook directory or any parent, so the notebook
# runs regardless of the working directory the kernel was started in.
RELATIVE_CANDIDATES = [
    Path('../data/spx_returns_weekly.xlsx'),
    Path('data/spx_returns_weekly.xlsx'),
    Path('spx_returns_weekly.xlsx'),
]
SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
DATA_PATH = next(
    (root / rel for root in SEARCH_ROOTS for rel in RELATIVE_CANDIDATES if (root / rel).is_file()),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError(
        'spx_returns_weekly.xlsx not found. Expected it at one of '
        + ', '.join(str(c) for c in RELATIVE_CANDIDATES)
        + ' relative to the notebook directory or a parent.'
    )

df_return = pd.read_excel(DATA_PATH, sheet_name='spx returns').set_index('date').sort_index()

TICKS = ['AAPL', 'META', 'NVDA', 'TSLA']
r = df_return[TICKS].apply(pd.to_numeric, errors='coerce').dropna(how='any')

q, ANN, M = 0.05, 52, 26   # significance level, weeks per year, rolling window
Z_05 = -1.65               # normal quantile approximation specified by the assignment

print(f'Workbook:      {DATA_PATH}')
print(f'Securities:    {", ".join(TICKS)}')
print(f'Sample:        {r.index[0]:%Y-%m-%d} to {r.index[-1]:%Y-%m-%d}  ({len(r)} weekly observations)')
print(f'Missing values after alignment: {int(r.isna().sum().sum())}')

Workbook:      /Users/samleung/Desktop/Uchicago/4th Year/FINM 250/Quant-Part-II/scripts/../data/spx_returns_weekly.xlsx
Securities:    AAPL, META, NVDA, TSLA
Sample:        2015-07-03 to 2026-07-03  (575 weekly observations)
Missing values after alignment: 0


In [2]:
df_return 

,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
date,,,,,,,,,,,,,,,,,,,,,
2015-07-03,0.008152,-0.001264,-0.004378,0.000000,-0.002220,-0.003352,-0.007444,-0.002409,-0.002943,2.078742e-04,...,-0.004997,0.004560,0.009847,0.009348,-0.002174,0.000000,-0.001528,-0.009341,-0.003368,-0.002478
2015-07-10,-0.004549,-0.024991,0.014953,0.000000,0.010525,0.014916,0.014898,-0.001981,-0.044579,-2.960025e-02,...,-0.017277,0.003477,0.022240,-0.011065,-0.036483,0.000000,-0.007537,-0.012755,-0.023837,-0.027945
2015-07-17,0.013958,0.051426,0.018481,0.000000,0.004226,-0.001441,0.017209,0.018737,0.015444,1.715711e-02,...,-0.025888,-0.021078,0.004474,0.004744,0.011868,0.000000,-0.028467,0.006460,0.040364,0.017888
2015-07-24,-0.016019,-0.039501,-0.027289,0.000000,0.023046,0.013994,0.006868,-0.013642,-0.063560,-1.728305e-02,...,-0.023294,0.007080,-0.024926,-0.032320,-0.032951,0.000000,-0.013313,-0.007814,-0.028814,0.035356
2015-07-31,0.041719,-0.025701,0.028349,0.000000,-0.007053,0.015512,0.019175,0.012472,-0.002736,4.872236e-03,...,0.030904,0.007810,0.055082,-0.009132,-0.002888,0.000000,0.012110,-0.024284,-0.029399,-0.010305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-05,-0.000664,-0.015125,0.043680,0.001725,0.063902,0.020707,-0.047148,-0.029976,-0.027451,1.428930e-02,...,0.007341,0.032207,-0.005785,0.032080,0.003652,-0.099974,0.019736,0.060732,-0.047285,0.022525
2026-06-12,-0.041347,-0.052743,0.002200,-0.009435,-0.031734,0.005154,-0.044712,-0.188594,0.040858,-8.403361e-03,...,0.015114,0.026704,0.002277,-0.019410,0.001273,0.020103,0.022801,0.014199,-0.015898,0.001636
2026-06-19,-0.021411,0.023632,-0.049357,0.076580,0.002608,-0.005237,-0.248414,-0.043427,0.039900,-6.405783e-02,...,-0.021328,-0.016221,-0.015484,-0.062581,0.012173,0.075662,-0.015035,-0.006774,0.033097,-0.010808


---
# 1. Diversification

## 1.1 Unconditional statistics, full sample

For each security, over the full sample:

- **volatility** — weekly standard deviation (`ddof=1`), also shown annualized by $\sqrt{52}$
- **empirical VaR (0.05)** — the direct $5\%$ sample quantile, `.quantile(0.05)`
- **empirical CVaR (0.05)** — the mean of the returns at or below that quantile

In [ ]:
def stats(x):
    """Unconditional volatility, empirical VaR and empirical CVaR of a return series."""
    v = x.std(ddof=1)                 # weekly volatility
    var = x.quantile(q)               # empirical VaR: direct 5% sample quantile
    cvar = x[x <= var].mean()          # empirical CVaR: mean of the tail at or beyond the quantile
    return pd.Series({'vol_wk': v, 'vol_ann': v * np.sqrt(ANN), 'VaR': var, 'CVaR': cvar})


risk_metrics_tbl = r.apply(stats).T
display(risk_metrics_tbl.style.format('{:.2%}').set_caption('1.1 Individual securities, full sample'))

,vol_wk,vol_ann,VaR,CVaR
AAPL,3.86%,27.85%,-5.48%,-8.30%
META,4.96%,35.76%,-7.23%,-10.64%
NVDA,6.30%,45.40%,-8.58%,-11.53%
TSLA,8.03%,57.92%,-11.70%,-14.78%


## 1.2 Equally-weighted portfolio

The equally-weighted portfolio return is the cross-sectional average of the four weekly returns,
$r^p_t = \tfrac14\sum_{i=1}^{4} r_{i,t}$. The statistics of 1.1 are recomputed on that series and
shown next to the individual securities.

To identify what drives the result, the realized ratio

$$\frac{\sigma_p}{\overline{\sigma_i}}$$

is compared to the value predicted by the equal-volatility, equal-correlation approximation

$$\frac{\sigma_p}{\sigma} = \sqrt{\frac{1}{N} + \left(1 - \frac{1}{N}\right)\bar{\rho}}
\qquad\text{with } N = 4 .$$

In [ ]:
timeline = r.mean(axis=1)          # equal weights -> simple average return each week
timeline_tbl = stats(timeline)

comparison_11_12 = pd.concat(
    [risk_metrics_tbl, timeline_tbl.to_frame('Equal-weighted').T],
)
comparison_11_12.loc['Average of the four securities'] = risk_metrics_tbl.mean()
display(
    comparison_11_12.style.format('{:.2%}')
    .set_caption('1.2 Equally-weighted portfolio vs. the individual securities')
)

# What is driving the result: average pairwise correlation.
corr = r.corr()
rho_bar = corr.values[np.triu_indices(len(TICKS), 1)].mean()   # mean of the upper triangle

N = len(TICKS)
predicted_ratio = np.sqrt(1 / N + (1 - 1 / N) * rho_bar)
realized_ratio = timeline_tbl['vol_wk'] / risk_metrics_tbl['vol_wk'].mean()

display(corr.style.format('{:.3f}').set_caption('Correlation matrix, full sample'))
print(f'Average pairwise correlation      rho_bar = {rho_bar:.4f}')
print(f'Average standalone weekly vol             = {risk_metrics_tbl["vol_wk"].mean():.4%}')
print(f'Portfolio weekly vol                      = {timeline_tbl["vol_wk"]:.4%}')
print(f'Predicted vol ratio  sqrt(1/N+(1-1/N)rho) = {predicted_ratio:.4f}')
print(f'Realized  vol ratio  sigma_p / mean(sigma)= {realized_ratio:.4f}')

,vol_wk,vol_ann,VaR,CVaR
AAPL,3.86%,27.85%,-5.48%,-8.30%
META,4.96%,35.76%,-7.23%,-10.64%
NVDA,6.30%,45.40%,-8.58%,-11.53%
TSLA,8.03%,57.92%,-11.70%,-14.78%
Equal-weighted,4.33%,31.20%,-6.06%,-8.42%
Average of the four securities,5.79%,41.73%,-8.25%,-11.31%


,AAPL,META,NVDA,TSLA
AAPL,1.000,0.408,0.480,0.437
META,0.408,1.000,0.427,0.265
NVDA,0.480,0.427,1.000,0.415
TSLA,0.437,0.265,0.415,1.000


Average pairwise correlation      rho_bar = 0.4055
Average standalone weekly vol             = 5.7870%
Portfolio weekly vol                      = 4.3266%
Predicted vol ratio  sqrt(1/N+(1-1/N)rho) = 0.7444
Realized  vol ratio  sigma_p / mean(sigma)= 0.7476


### 1.2 — What do you find, and what is driving it?

The portfolio is materially less risky than the average of its holdings, but it is **not** less risky than
its safest holding.

| | weekly vol | annualized vol | VaR (0.05) | CVaR (0.05) |
|---|---|---|---|---|
| Average of the four securities | 5.79% | 41.73% | −8.25% | −11.31% |
| Equally-weighted portfolio | 4.33% | 31.20% | −6.06% | −8.42% |
| Safest single security (AAPL) | 3.86% | 27.85% | −5.48% | −8.30% |

Volatility falls to **4.33% from an average standalone 5.79%**, a ratio of **0.748**. Every portfolio
statistic sits inside the range spanned by the individual securities — well below TSLA, still above AAPL.

The driver is **imperfect correlation**, and the arithmetic is exact. With $N=4$ roughly comparable
volatilities and an average pairwise correlation of $\bar\rho = 0.406$, theory predicts the portfolio
retains $\sqrt{0.25 + 0.75 \times 0.406} = \mathbf{0.744}$ of the average standalone volatility. The data
delivers $\mathbf{0.748}$. The two agree to within four basis points of a ratio, so the ~25% risk
reduction is entirely accounted for by diversification across correlated assets — nothing else is needed
to explain it.

Note where the benefit comes from and where it stops. The $1/N$ term is the idiosyncratic risk that
averaging destroys; the $\bar\rho$ term is the common market factor that averaging cannot touch. Because
$\bar\rho \approx 0.41$ rather than 0, roughly three-quarters of the average volatility survives. Adding
more names of this type would push the ratio toward $\sqrt{\bar\rho} = 0.64$, not toward zero — these are
four large-cap US technology stocks loaded on the same factor, so diversification here has a hard floor.

## 1.3 Drop the most volatile asset, hold the freed weight in cash

The most volatile security from 1.1 is dropped and its 25% weight is held in cash earning a return of
zero (a negligibly small risk-free rate). The remaining three securities keep their 25% weights, so the
portfolio return is

$$r^{p,3}_t = 0.25 \sum_{i \in \text{keep}} r_{i,t} + 0.25 \times 0 .$$

To answer whether the change is in line with the stand-alone risk assessment of 1.1, the removed asset's
share of risk is measured two ways:

1. **Stand-alone share** — its volatility divided by the sum of the four stand-alone volatilities, which
   is what 1.1 alone would suggest.
2. **Component contribution to risk** — the marginal decomposition
   $\text{CCR}_i = w_i (\Sigma w)_i / \sigma_p$, which sums exactly to $\sigma_p$ and does account for
   correlations.

In [ ]:
worst = risk_metrics_tbl['vol_wk'].idxmax()
keep = [t for t in TICKS if t != worst]

three_stock_portfolio = r[keep].sum(axis=1) * 0.25   # 25% in each survivor, 25% in cash at 0%
three_stock_tbl = stats(three_stock_portfolio)

print(f'Most volatile asset dropped: {worst} '
      f'(weekly vol {risk_metrics_tbl.loc[worst, "vol_wk"]:.2%})')
print(f'Held: {", ".join(keep)} at 25% each, plus 25% cash at 0%')

comparison_12_13 = pd.DataFrame({
    '1.2 four securities': timeline_tbl,
    f'1.3 drop {worst}, 25% cash': three_stock_tbl,
}).T
comparison_12_13['change'] = comparison_12_13['vol_wk'] / timeline_tbl['vol_wk'] - 1
display(
    comparison_12_13.style.format({'vol_wk': '{:.2%}', 'vol_ann': '{:.2%}',
                                   'VaR': '{:.2%}', 'CVaR': '{:.2%}', 'change': '{:+.1%}'})
    .set_caption(f'1.3 Portfolio with {worst} replaced by cash')
)

vol_drop = 1 - three_stock_tbl['vol_wk'] / timeline_tbl['vol_wk']
var_drop = 1 - (three_stock_tbl['vol_wk'] / timeline_tbl['vol_wk']) ** 2
print(f'\nVolatility reduction from removing {worst}: {vol_drop:.2%}')
print(f'Variance   reduction from removing {worst}: {var_drop:.2%}')

Most volatile asset dropped: TSLA (weekly vol 8.03%)
Held: AAPL, META, NVDA at 25% each, plus 25% cash at 0%


,vol_wk,vol_ann,VaR,CVaR,change
1.2 four securities,4.33%,31.20%,-6.06%,-8.42%,+0.0%
"1.3 drop TSLA, 25% cash",3.01%,21.69%,-4.22%,-6.00%,-30.5%



Volatility reduction from removing TSLA: 30.47%
Variance   reduction from removing TSLA: 51.65%


In [ ]:
# Is that in line with the stand-alone risk assessment of 1.1?
weight = np.repeat(1 / len(TICKS), len(TICKS))   # equal 25% weights
sigma = r.cov().values                           # covariance matrix of the four return series
total_portfolio_vol = np.sqrt(weight @ sigma @ weight)

# Component contribution to risk: w_i * (Sigma w)_i / sigma_p, which sums to sigma_p.
component_contribution_risk = weight * (sigma @ weight) / total_portfolio_vol
pct_contrib = component_contribution_risk / total_portfolio_vol

standalone_share = risk_metrics_tbl['vol_wk'] / risk_metrics_tbl['vol_wk'].sum()

risk_share_tbl = pd.DataFrame({
    'capital weight': weight,
    'stand-alone vol': risk_metrics_tbl['vol_wk'].values,
    'stand-alone share (1.1 view)': standalone_share.values,
    'component contribution to risk': pct_contrib,
}, index=TICKS)

display(
    risk_share_tbl.style.format({'capital weight': '{:.0%}', 'stand-alone vol': '{:.2%}',
                                 'stand-alone share (1.1 view)': '{:.1%}',
                                 'component contribution to risk': '{:.1%}'})
    .set_caption('Risk shares of the equally-weighted portfolio')
)

assert np.isclose(component_contribution_risk.sum(), total_portfolio_vol), \
    'Component contributions must sum to portfolio volatility.'
assert np.isclose(total_portfolio_vol, timeline_tbl['vol_wk']), \
    'Covariance-based portfolio vol must match the vol of the averaged return series.'

print(f'Portfolio weekly vol (covariance form) = {total_portfolio_vol:.4%}')
print(f'{worst}: stand-alone share {standalone_share[worst]:.1%} vs. '
      f'component contribution {pct_contrib[TICKS.index(worst)]:.1%}')

,capital weight,stand-alone vol,stand-alone share (1.1 view),component contribution to risk
AAPL,25%,3.86%,16.7%,16.0%
META,25%,4.96%,21.4%,18.8%
NVDA,25%,6.30%,27.2%,28.6%
TSLA,25%,8.03%,34.7%,36.6%


Portfolio weekly vol (covariance form) = 4.3266%
TSLA: stand-alone share 34.7% vs. component contribution 36.6%


### 1.3 — Is this in line with the stand-alone risk assessment?

**In ranking, yes. In magnitude, no — and the gap is informative.**

Replacing TSLA with cash cuts portfolio weekly volatility from **4.33% to 3.01%**, a **30.5% reduction**;
annualized volatility falls from 31.20% to 21.69%, VaR from −6.06% to −4.22%, and CVaR from −8.42% to
−6.00%. TSLA was the largest single source of risk, exactly as 1.1 implied.

But the stand-alone view of 1.1 **understates** how much risk TSLA was carrying:

| | TSLA |
|---|---|
| Capital weight | 25.0% |
| Stand-alone vol | 8.03% weekly (highest of the four) |
| Stand-alone share of summed vols (1.1 view) | 34.7% |
| Component contribution to portfolio risk | **36.6%** |

The 1.1 view treats each security in isolation and so ignores TSLA's ~0.40 correlation with the other
three. Once covariances are counted, TSLA bears **36.6% of portfolio risk on 25% of the capital**.

The 30.5% volatility drop is **not directly comparable** to that 36.6%, and the two should not be expected
to match. Two distinct effects separate them:

1. **Marginal vs. discrete.** Component contribution splits each covariance term $2 w_i w_j \sigma_{ij}$
   evenly between the two assets involved. Deleting the position removes the *whole* term. So removal
   eliminates **51.65% of portfolio variance** — more than TSLA's 36.6% marginal share.
2. **The square root.** Volatility is the square root of variance, which compresses a 51.65% variance drop
   into $1 - \sqrt{1 - 0.5165} = 30.5\%$ in volatility units.

So: marginal risk decomposition and discrete removal answer different questions. They agree that TSLA was
the dominant risk contributor and that its influence exceeded its capital weight; they disagree on the
size of the number, by construction. The practical reading is that a 25% cash allocation bought a 30.5%
reduction in volatility — a favorable trade only because the asset it displaced was carrying an
outsized, correlation-amplified share of the risk.

---
# 2. Dynamic Measures

## 2.1 Conditional statistics at the end of the sample

This section uses the equally-weighted portfolio of **1.2**. Returns are weekly, so the rolling window is
$M = 26$ weeks and volatility is annualized with $\sqrt{52}$.

**Volatility.** The rolling volatility series is computed for each security and for the portfolio. Note
that the rolling standard deviation of the equally-weighted return series is *identical* to
$\sqrt{w'\hat\Sigma_t w}$ built from the rolling covariance matrix over the same window; this is asserted
below rather than assumed.

**Mean.** The conditional mean return is approximated as zero.

**Notation.** Following the notes, $\sigma_t$ is estimated from data through $t-1$ and used to forecast
period $t$. Therefore the estimate formed from the $M$ observations ending at the final date $T$ is
$\sigma_{T+1}$ — the forecast one would act on standing at the end of the sample. That is the value used
for the reported VaR and CVaR.

"At the end of the sample" admits a second reading, in which the requested quantity is $\sigma_T$ (the
window ending at $T-1$, the last forecast the backtest in 2.2 can actually score). The two differ by a
single weekly observation, and a sensitivity table below reports both: annualized volatility of 25.35%
versus 24.65%, VaR of −5.80% versus −5.64%. **Every conclusion drawn below holds under either
convention.**

**VaR and CVaR.** With $\mu = 0$ and $z_{0.05} = -1.65$ as specified:

$$\text{VaR}_{0.05} = z_{0.05}\,\sigma
\qquad
\text{CVaR}_{0.05} = -\,\sigma\,\frac{\phi(z_{0.05})}{0.05} \approx -2.045\,\sigma$$

Both are reported as returns, so losses are negative.

In [ ]:
equal_weighted_return = r.mean(axis=1)

# Each value dated t uses the M observations ending at t. At the final date T this is
# the forecast for T+1, i.e. sigma_{T+1} in the notation of the notes.
rolling_vol_security = r.rolling(M).std(ddof=1)
rolling_vol_portfolio = equal_weighted_return.rolling(M).std(ddof=1)

# Cross-check: rolling vol of the averaged series == sqrt(w' Sigma_roll w) on the same window.
_w = np.repeat(1 / len(TICKS), len(TICKS))
_sigma_roll_end = r.rolling(M).cov().loc[r.index[-1]].values
assert np.isclose(np.sqrt(_w @ _sigma_roll_end @ _w), rolling_vol_portfolio.iloc[-1]), \
    'Portfolio rolling vol must equal the covariance-matrix form.'

sigma_forecast = rolling_vol_portfolio.iloc[-1]
normal_var_05 = Z_05 * sigma_forecast
normal_cvar_05 = -(norm.pdf(Z_05) / q) * sigma_forecast

end_security_vol = pd.DataFrame({
    'weekly volatility': rolling_vol_security.iloc[-1],
    'annualized volatility': rolling_vol_security.iloc[-1] * np.sqrt(ANN),
})

print(f'End of sample:  {equal_weighted_return.index[-1]:%Y-%m-%d}')
print(f'Rolling window: {M} weeks '
      f'({rolling_vol_portfolio.index[-M]:%Y-%m-%d} to {rolling_vol_portfolio.index[-1]:%Y-%m-%d})')
print(f'Normal CVaR multiplier  -phi(z)/q = {-norm.pdf(Z_05) / q:.4f}')

display(end_security_vol.style.format('{:.2%}')
        .set_caption(f'End-of-sample {M}-week volatility by security'))

section_21 = pd.Series({
    'volatility (annualized)': sigma_forecast * np.sqrt(ANN),
    'normal VaR (0.05)': normal_var_05,
    'normal CVaR (0.05)': normal_cvar_05,
}, name='Section 2.1')
display(section_21.to_frame('value').style.format('{:.2%}')
        .set_caption('2.1 Requested portfolio measures (conditional, end of sample)'))

End of sample:  2026-07-03
Rolling window: 26 weeks (2026-01-09 to 2026-07-03)
Normal CVaR multiplier  -phi(z)/q = -2.0453


,weekly volatility,annualized volatility
AAPL,3.96%,28.59%
META,5.93%,42.79%
NVDA,4.33%,31.24%
TSLA,5.08%,36.60%


,value
volatility (annualized),25.35%
normal VaR (0.05),-5.80%
normal CVaR (0.05),-7.19%


In [ ]:
# Compare with the unconditional answers of 1.2.
comparison_12_21 = pd.DataFrame({
    '1.2 full-sample empirical': [
        timeline_tbl['vol_ann'], timeline_tbl['VaR'], timeline_tbl['CVaR'],
    ],
    '2.1 conditional normal': [
        section_21['volatility (annualized)'],
        section_21['normal VaR (0.05)'],
        section_21['normal CVaR (0.05)'],
    ],
}, index=['Volatility (annualized)', 'VaR (0.05)', 'CVaR (0.05)'])
comparison_12_21['difference (2.1 - 1.2)'] = (
    comparison_12_21['2.1 conditional normal'] - comparison_12_21['1.2 full-sample empirical']
)
display(comparison_12_21.style.format('{:.2%}').set_caption('2.1 vs. 1.2'))

# Where does the current window sit in the history of the rolling volatility series?
vol_pctile = (rolling_vol_portfolio.dropna() <= sigma_forecast).mean()
print(f'Rolling vol at end of sample:  {sigma_forecast:.2%} weekly / '
      f'{sigma_forecast * np.sqrt(ANN):.2%} annualized')
print(f'Full-sample unconditional vol: {timeline_tbl["vol_wk"]:.2%} weekly / '
      f'{timeline_tbl["vol_ann"]:.2%} annualized')
print(f'Percentile of the end-of-sample window within the rolling vol history: {vol_pctile:.1%}')
print(f'Rolling vol range: {rolling_vol_portfolio.min():.2%} to '
      f'{rolling_vol_portfolio.max():.2%} weekly')

,1.2 full-sample empirical,2.1 conditional normal,difference (2.1 - 1.2)
Volatility (annualized),31.20%,25.35%,-5.85%
VaR (0.05),-6.06%,-5.80%,0.26%
CVaR (0.05),-8.42%,-7.19%,1.23%


Rolling vol at end of sample:  3.52% weekly / 25.35% annualized
Full-sample unconditional vol: 4.33% weekly / 31.20% annualized
Percentile of the end-of-sample window within the rolling vol history: 36.8%
Rolling vol range: 2.26% to 8.03% weekly


In [ ]:
# Robustness to the reading of "at the end of the sample". The window either ends at T
# (forecasting T+1, the position one holds standing at the end of the sample) or at T-1
# (sigma_T under the notes' dating, the last forecast that 2.2 can actually score).
def normal_measures(sigma_wk):
    return {'sigma (weekly)': sigma_wk,
            'volatility (annualized)': sigma_wk * np.sqrt(ANN),
            'normal VaR (0.05)': Z_05 * sigma_wk,
            'normal CVaR (0.05)': -(norm.pdf(Z_05) / q) * sigma_wk}


timing_check = pd.DataFrame([
    normal_measures(rolling_vol_portfolio.iloc[-1]),
    normal_measures(rolling_vol_portfolio.shift(1).iloc[-1]),
], index=[
    f'Window ends {rolling_vol_portfolio.index[-1]:%Y-%m-%d} (T)   -> forecast for T+1  [reported above]',
    f'Window ends {rolling_vol_portfolio.index[-2]:%Y-%m-%d} (T-1) -> forecast for T',
])
display(timing_check.style.format('{:.2%}')
        .set_caption('2.1 Sensitivity to the end-of-sample dating convention'))

print('The two conventions differ by one weekly observation: '
      f'{abs(timing_check["volatility (annualized)"].diff().iloc[-1]) * 100:.2f} pp of annualized '
      'volatility. The conclusions in the discussion below are unchanged either way.')

,sigma (weekly),volatility (annualized),normal VaR (0.05),normal CVaR (0.05)
Window ends 2026-07-03 (T) -> forecast for T+1 [reported above],3.52%,25.35%,-5.80%,-7.19%
Window ends 2026-06-26 (T-1) -> forecast for T,3.42%,24.65%,-5.64%,-6.99%


The two conventions differ by one weekly observation: 0.70 pp of annualized volatility. The conclusions in the discussion below are unchanged either way.


### 2.1 — How do these compare to 1.2?

| | 1.2 unconditional, empirical | 2.1 conditional, normal | difference |
|---|---|---|---|
| Volatility (annualized) | 31.20% | **25.35%** | −5.85 pp |
| VaR (0.05) | −6.06% | **−5.80%** | +0.26 pp |
| CVaR (0.05) | −8.42% | **−7.19%** | +1.23 pp |

**All three conditional measures are smaller.** Two separate causes are at work, and they should be kept
apart.

**1. The conditioning information.** The last 26 weeks were calmer than the sample as a whole: the
forecast volatility is 25.35% annualized against an unconditional 31.20%, sitting at the **36.8th
percentile** of the rolling volatility history, which spans 16.3% to 57.9% annualized. This is what a
conditional measure is *for* — 1.2 answers "how risky has this portfolio been on average since 2015",
while 2.1 answers "how risky is it right now". A one-week-ahead forecast should not be anchored to a
decade-long average when volatility clusters as strongly as it does here: the autocorrelation of squared
weekly portfolio returns is 0.24 at lag 1, so a calm recent window genuinely carries information about
next week.

**2. The distributional assumption.** The normal assumption bites hardest in the tail, and the CVaR gap is
the tell. The ratio CVaR/VaR is **1.24** under normality ($2.045/1.65$, fixed by the assumption) but
**1.39** in the empirical 1.2 numbers ($-8.42/-6.06$). The realized tail is fatter than Gaussian. So even
after adjusting for the calmer window, a normal CVaR understates the severity of a bad week; the
25.35%-vs-31.20% volatility gap explains most of the difference, but not the extra tail thickness.

**Reading the two together:** the conditional numbers are the right ones to act on for the coming week
because volatility is persistent and currently below average. The caveat is that VaR and especially CVaR
computed under normality will be too optimistic in the tail whatever the volatility input — which is
precisely what the backtest in 2.2 tests directly.

## 2.2 VaR hit-test backtest

A **hit** occurs when the realized portfolio return at $t$ falls below the VaR forecast for $t$:

$$\text{hit}_t = \mathbf{1}\!\left\{ r^p_t < z_{0.05}\,\sigma_t \right\}$$

The forecast applied to $t$ must use only information available through $t-1$, so both volatility
estimates are **shifted forward one period** before the comparison. Without the shift, the realized return
being tested would be inside its own volatility estimate — a look-ahead that biases the hit rate downward.

Two volatility estimators are compared:

- **expanding** — all returns through $t-1$
- **rolling** — the $M = 26$ returns through $t-1$

The two estimators become available on different dates (the expanding one starts almost immediately, the
rolling one only after 26 weeks), so a **common sample** is also reported. Only the common-sample rows
compare the estimators on equal footing; the full-sample expanding rate additionally includes early dates
where the estimate rests on a handful of observations.

A correctly specified 5% VaR should be hit about 5% of the time.

In [ ]:
# The shift is essential: the forecast applied to return t uses data only through t-1.
expanding_vol_forecast = equal_weighted_return.expanding(min_periods=2).std(ddof=1).shift(1)
rolling_vol_forecast = equal_weighted_return.rolling(M).std(ddof=1).shift(1)

expanding_var = Z_05 * expanding_vol_forecast
rolling_var = Z_05 * rolling_vol_forecast


def hit_test(realized, var_forecast, mask=None):
    """Hit count and rate over the dates where the forecast exists (optionally restricted)."""
    valid = var_forecast.notna() if mask is None else var_forecast.notna() & mask
    hits = realized.loc[valid] < var_forecast.loc[valid]
    n = int(valid.sum())
    return {'forecast observations': n,
            'expected hits at 5%': q * n,
            'number of hits': int(hits.sum()),
            'percentage of hits': hits.mean()}


common = expanding_var.notna() & rolling_var.notna()

hit_test_results = pd.DataFrame([
    hit_test(equal_weighted_return, expanding_var),
    hit_test(equal_weighted_return, rolling_var),
    hit_test(equal_weighted_return, expanding_var, common),
    hit_test(equal_weighted_return, rolling_var, common),
], index=[
    'Expanding volatility (full)',
    f'Rolling volatility ({M} weeks, full)',
    'Expanding volatility (common sample)',
    f'Rolling volatility ({M} weeks, common sample)',
])

display(hit_test_results.style.format({
    'forecast observations': '{:.0f}',
    'expected hits at 5%': '{:.1f}',
    'number of hits': '{:.0f}',
    'percentage of hits': '{:.2%}',
}).set_caption('2.2 VaR(0.05) hit test, forecasts using information through t-1'))

,forecast observations,expected hits at 5%,number of hits,percentage of hits
Expanding volatility (full),572,28.6,26,4.55%
"Rolling volatility (26 weeks, full)",548,27.4,22,4.01%
Expanding volatility (common sample),548,27.4,24,4.38%
"Rolling volatility (26 weeks, common sample)",548,27.4,22,4.01%


In [ ]:
# Diagnostic: why do both estimators under-hit? Test the mu = 0 assumption by restoring
# the conditional mean, keeping everything else identical.
mu_expanding = equal_weighted_return.expanding(min_periods=2).mean().shift(1)
mu_rolling = equal_weighted_return.rolling(M).mean().shift(1)

drift_check = pd.DataFrame([
    hit_test(equal_weighted_return, expanding_var, common),
    hit_test(equal_weighted_return, mu_expanding + expanding_var, common),
    hit_test(equal_weighted_return, rolling_var, common),
    hit_test(equal_weighted_return, mu_rolling + rolling_var, common),
], index=[
    'Expanding, mu = 0 (as specified)',
    'Expanding, mu restored',
    'Rolling, mu = 0 (as specified)',
    'Rolling, mu restored',
])
display(drift_check[['number of hits', 'percentage of hits']].style.format({
    'number of hits': '{:.0f}', 'percentage of hits': '{:.2%}',
}).set_caption('Effect of the mu = 0 assumption on the hit rate (common sample)'))

hit_se = np.sqrt(q * (1 - q) / int(common.sum()))
print(f'Mean weekly portfolio return: {equal_weighted_return.mean():.4%} '
      f'({equal_weighted_return.mean() * ANN:.2%} annualized) -- not zero')
print(f'Volatility clustering, autocorrelation of squared returns at lag 1: '
      f'{equal_weighted_return.pow(2).autocorr(1):.3f}')
print(f'Standard error of a 5% hit rate over {int(common.sum())} observations: {hit_se:.2%}')

,number of hits,percentage of hits
"Expanding, mu = 0 (as specified)",24,4.38%
"Expanding, mu restored",30,5.47%
"Rolling, mu = 0 (as specified)",22,4.01%
"Rolling, mu restored",32,5.84%


Mean weekly portfolio return: 0.7588% (39.46% annualized) -- not zero
Volatility clustering, autocorrelation of squared returns at lag 1: 0.243
Standard error of a 5% hit rate over 548 observations: 0.93%


In [ ]:
# Validation: confirm the forecasts really only use information through t-1.
first_expanding_date = expanding_vol_forecast.first_valid_index()
pos = equal_weighted_return.index.get_loc(first_expanding_date)
assert np.isclose(
    expanding_vol_forecast.loc[first_expanding_date],
    equal_weighted_return.iloc[:pos].std(ddof=1),
), 'Expanding volatility forecast is not aligned with information through t-1.'

first_rolling_date = rolling_vol_forecast.first_valid_index()
pos = equal_weighted_return.index.get_loc(first_rolling_date)
assert np.isclose(
    rolling_vol_forecast.loc[first_rolling_date],
    equal_weighted_return.iloc[pos - M:pos].std(ddof=1),
), 'Rolling volatility forecast is not aligned with the previous M observations.'

# The unshifted series would leak the return being tested into its own forecast.
assert not np.isclose(
    rolling_vol_forecast.iloc[-1],
    equal_weighted_return.rolling(M).std(ddof=1).iloc[-1],
), 'Shift did not take effect.'

# The common sample must be exactly the dates where the rolling forecast exists.
assert common.equals(rolling_var.notna()), 'Common sample should equal the rolling forecast sample.'
assert int(common.sum()) == len(equal_weighted_return) - M

print('Validation passed:')
print('  - both VaR forecasts use only information through t-1')
print(f'  - first expanding forecast: {first_expanding_date:%Y-%m-%d}')
print(f'  - first rolling forecast:   {first_rolling_date:%Y-%m-%d}')
print(f'  - common sample: {int(common.sum())} of {len(equal_weighted_return)} weeks')

Validation passed:
  - both VaR forecasts use only information through t-1
  - first expanding forecast: 2015-07-24
  - first rolling forecast:   2016-01-08
  - common sample: 548 of 574 weeks


### 2.2 — Percentage of hits

| estimator | sample | obs | expected hits | actual hits | **hit rate** |
|---|---|---|---|---|---|
| Expanding | full | 572 | 28.6 | 26 | **4.55%** |
| Rolling (26w) | full | 548 | 27.4 | 22 | **4.01%** |
| Expanding | common | 548 | 27.4 | 24 | **4.38%** |
| Rolling (26w) | common | 548 | 27.4 | 22 | **4.01%** |

**Both estimators are mildly conservative — they produce fewer hits than the 5% they promise.** On the
common sample the expanding estimator hits **4.38%** and the rolling estimator **4.01%**, against 27.4
expected breaches versus 24 and 22 actual.

Note that the headline full-sample rates (4.55% vs. 4.01%) are **not** a like-for-like comparison: the
expanding estimator is available from week 3 while the rolling estimator needs 26 weeks, so the two cover
different samples. Restricting both to the 548 common weeks narrows the gap from 0.54 pp to 0.37 pp. The
expanding estimator is still the closer of the two to 5%, so the ranking survives — but the true
difference is about a third smaller than the naive comparison suggests, and two of the expanding
estimator's 26 full-sample hits come from early dates where the volatility estimate rested on only a
handful of observations.

**Why under-hitting rather than over-hitting?** Two forces push in opposite directions:

- $z_{0.05} = -1.65$ with a *normal* assumption should **under**-state a fat left tail, pushing the hit
  rate above 5%.
- The $\mu = 0$ assumption discards a large positive drift. This portfolio averages **+0.76% per week**
  (39.5% annualized) over 2015–2026. Setting $\mu = 0$ places the threshold at $-1.65\sigma$ instead of
  $\mu - 1.65\sigma$, i.e. further below where returns actually sit, making breaches *less* frequent.

**The dropped drift dominates**, and the diagnostic above isolates it: restoring the conditional mean and
changing nothing else moves the rolling hit rate from **4.01% to 5.84%** and the expanding rate from 4.38%
to 5.47% on the common sample. So the under-hitting is an artifact of the $\mu = 0$ simplification, not
evidence that the volatility model is too conservative. Once the drift is restored the rates overshoot 5%
slightly — which is the fat tail finally showing through, consistent with the CVaR/VaR ratio in 2.1.

Neither miss is large in statistical terms. With 548 observations the standard error of a 5% hit rate is
$\sqrt{0.05 \times 0.95 / 548} = 0.93$ pp, so 4.01% is about one standard error from 5% and 4.38% well
inside that. **Neither estimator is rejected** on the unconditional coverage test; the specification is
adequately calibrated with a mild conservative bias traceable to the zero-mean assumption.

**Which is preferable?** The rolling estimator is further from 5% unconditionally, but that is the weaker
test. The expanding estimator converges to a constant unconditional volatility and cannot respond when
regimes shift, so its breaches should arrive in clusters during volatile stretches — a violation of the
independence half of correct VaR specification even when the count looks right. The rolling window adapts,
which is the property that matters for a one-week-ahead forecast, and it is exactly the estimator used to
produce the 2.1 numbers.